## Mamogram analysis

[Nationwide real-world implementation of AI for cancer detection in population-based mammography screening](https://www.nature.com/articles/s41591-024-03408-6)

Download the praim.csv from here.
https://datadryad.org/dataset/doi:10.5061/dryad.zs7h44jgn

In [ ]:
import pandas as pd

mamogram_file = "~/Downloads/praim.csv"



In [ ]:
import random
def pipe_filter_exclude_ai_viewer(df):

   return df.query("used_ai_viewer == False").copy()


def compute_pd(df):
   #  first and second read are the same no consense ( 3rd label is NOT used)
   df["pd"] = df[["first_read",	"second_read"]].apply(lambda x: 1.0 if x.iloc[0] == x.iloc[1] else round(2/3,2), axis=1)
   df["pd_majority"] = df["Majority label"]
   
   # Now change the majority label ("NOT Match" when no alignment)  to either sus or normal based on whether the patient was recalled.
   df.loc[df['pd_majority'] == 'NOT MATCH', "pd_majority"] = df.loc[df['pd_majority'] == 'NOT MATCH']["had_recall"].apply(lambda x: "not-normal" if x else "normal" )
   df.loc[df['pd_majority'] == 'suspicious', "pd_majority"] = "not-normal"

   df["all_humans"]= df.apply(lambda x: list([x["first_read"],	x["second_read"],  x['pd_majority']]) if x["Majority label"] == "NOT MATCH"  else  list([x["first_read"],	x["second_read"]]) , axis=1)

   for i in range(5):
        df[f"another_human_{i}"] = df.apply(lambda x: random.choice( x["all_humans"] ), axis=1)
        # Rename value from suspicious to not-normal
        df.loc[df[f"another_human_{i}"] == 'suspicious', f"another_human_{i}"] = "not-normal"



   return df
    
df = pd.read_csv(mamogram_file).pipe(pipe_filter_exclude_ai_viewer).pipe(compute_pd)



In [ ]:
df.sample(n=10)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score


def report_by_pd(df, prediction_label = "ai_prediction", predictor_friendly_name="AI"):
    items = []
    scores_dict = classification_report(df["pd_majority"], df[prediction_label],  output_dict=True, zero_division=0.0)["not-normal"]
    scores_dict["PA"]     = "ALL"
    scores_dict["S"]=len(df)
    scores_dict["accuracy"] = accuracy_score(df["pd_majority"], df[prediction_label])

    items.append(scores_dict)
    
    st_groups = list(df["pd"].unique())

    for i in st_groups:



        d_subset  = df.query(f"pd ==  {i}")
        d = classification_report(d_subset["pd_majority"], d_subset[prediction_label],  output_dict=True, zero_division=0.0)      
        if "not-normal"  in d:
            d_treu = d["not-normal"]
            d_treu["PA"]= round(i, 2)
            d_treu["raw_PA"]= i
            d_treu["accuracy"] =accuracy_score(d_subset["pd_majority"], d_subset[prediction_label])
            d_treu["S"]= len(d_subset)
            items.append(d_treu)

    df_scores = pd.DataFrame(items)[["S","PA","accuracy", "precision", "recall", "f1-score", "support"]].round(2)
    df_scores["support"] = df_scores["support"].div(df_scores["S"]).apply(lambda x: "{:.3f}".format(x)) 

    df_scores = df_scores[["PA","S", "support", "f1-score", "precision", "recall", "accuracy"]]
    idx = pd.MultiIndex.from_tuples(
        [
         (predictor_friendly_name, c)  if c in ["accuracy", "precision", "recall", "f1-score"] else  ("common", c)   for c in df_scores.columns
        ],
        names=["predictor", "score_type"]
    )

    df_scores.columns= pd.MultiIndex.from_tuples(idx)
    return df_scores
    
            
df_scores_ai = report_by_pd(df)
df_scores_ai

In [ ]:
from functools import reduce

another_human_scores = []
for l in [l for l in list(df.columns) if l.startswith("another_human")]:

     another_human_scores.append(report_by_pd(df, l, l))

# Merge all DataFrames on the 'key_col' column
df_scores_human_all = reduce(lambda left, right: pd.merge(left, right, on=[("common", "PA" ), ("common", "S" ), ("common", "support" )], how='inner'), another_human_scores)

df_scores_human_all

In [ ]:
df_scores_human_mean = df_scores_human_all.loc[:, pd.IndexSlice["common", :]].copy()
for c in ["recall", "precision", "f1-score", "accuracy"]:
    df_scores_human_mean.loc[:,("another_human", c)] = df_scores_human_all.loc[:, pd.IndexSlice[:, c]].mean(axis=1)
    df_scores_human_mean.loc[:,("another_human", f"{c}_std")] = df_scores_human_all.loc[:, pd.IndexSlice[:,c]].std(axis=1)
    df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:.4f}",axis=1)

df_scores_human_mean

In [ ]:
index_columns = [("common", "PA" ), ("common", "S" ), ("common", "support" )]
df_scores_all = df_scores_human_mean.merge(df_scores_ai, on=index_columns)
df_scores_all = df_scores_all.set_index(index_columns)

expected_formula_map = {
    'f1-score': lambda pd, m: (2*m*pd)/(2*m*pd + 1-pd),
    "accuracy": lambda pd, m: pd,
    "recall": lambda pd, m: pd,
    "precision": lambda pd, m: (m*pd)/(m*pd + (1-m)* (1-pd)),
}

def expected_all_accuracy(listdict):
    num_sum = 0
    denom_sum = 0
    for t in listdict :
       if  t["p"] == "ALL": continue
       num_sum =  num_sum + t["s"]* t["p"]
       denom_sum=denom_sum+ t["s"]
    return num_sum/denom_sum

def expected_all_recall(listdict):
    num_sum = 0
    denom_sum = 0
    for t in listdict :
       if  t["p"] == "ALL": continue
       num_sum =  num_sum + t["s"]*t["p"]* t["m"]
       denom_sum=denom_sum+ t["s"]*t["m"]
    return num_sum/denom_sum

def expected_all_precision(listdict):
    num_sum = 0
    denom_sum = 0
    for t in listdict :

       if  t["p"] == "ALL": continue
       num_sum =  num_sum + t["s"]*t["p"]* t["m"]
       denom_sum = denom_sum + t["s"]*(t["p"]*t["m"]+ (1-t["p"])*(1-t["m"]))
    return num_sum/denom_sum

def expected_all_f1(listdict):

    numerator =  2* expected_all_precision(listdict)*expected_all_recall(listdict)
    denom =  expected_all_precision(listdict) + expected_all_recall(listdict)
    return numerator/denom

expected_all_formula_map = {
    'f1-score': expected_all_f1,
    "accuracy": expected_all_accuracy,
    "recall": expected_all_recall,
    "precision": expected_all_precision,
}

def extract_pd_s_m(df):
    return [{"p": x[0], "s": x[1], "m": float(x[2])}
            for x in df.index]

for s in ["f1-score", "precision", "recall", "accuracy" ]:
    df_scores_all[("delta_h_ai", s)] = df_scores_all[("another_human", s )]  - df_scores_all[("AI", s )]

for s in ["f1-score", "precision", "recall", "accuracy" ]:

    df_scores_all[("Expected", s)] = list(pd.DataFrame(list(df_scores_all.index)).apply(lambda x: expected_formula_map[s](float(x[0]), float(x[2])) if x[0] != 'ALL' else expected_all_formula_map[s](extract_pd_s_m(df_scores_all)), axis=1 ))

    # df_scores_all[("Expected", s)] = list(pd.DataFrame(list(df_scores_all.index)).apply(lambda x: expected_formula_map[s](float(x[0]), float(x[2])) if x[0] != 'ALL' else "-", axis=1 ))

    df_scores_all[("delta_e_h", s)] = df_scores_all[("Expected", s)]  - df_scores_all[("another_human", s )]

    df_scores_all[("delta_e_ai", s)] = df_scores_all[("Expected", s)]  - df_scores_all[("AI", s )]

df_scores_all

In [ ]:


def _prep_display(df_scores):
    display_set_cols = []
    cols_score_types = ['f1-score', "accuracy" ]

    ai_predictor = "AI"
    human_predictor = "another_human"

    for score_type in cols_score_types:
        display_set_cols.append((ai_predictor, score_type))
        display_set_cols.append((human_predictor, f"fmt_{score_type}_meanstd"))
        display_set_cols.append(("Expected", score_type))
        display_set_cols.append(("delta_h_ai", score_type))
        display_set_cols.append(("delta_e_h", score_type))
        display_set_cols.append(("delta_e_ai", score_type))

    return df_scores[display_set_cols].sort_index()

_prep_display(df_scores_all )




In [ ]:
print(_prep_display(df_scores_all).reset_index().to_latex(float_format="{:.2f}".format, index=False))